# L2/3 Barrel Cortex Simulation Example

This notebook demonstrates how to build, simulate, and analyze a biorealistic L2/3 cortical network.

## Setup

In [ ]:
import sys
sys.path.append('/app')

import numpy as np
import matplotlib.pyplot as plt
from neuron import h

print(f"NEURON version: {h.nrnversion()}")
print("Setup complete!")

## 1. Build Network

In [ ]:
from network.build_network import build_l23_network

# Build a small network for quick testing
network = build_l23_network(n_cells=200, seed=42)

print(f"\nNetwork built with {len(network.cells)} cells")

## 2. Run Baseline Simulation

In [ ]:
from simulations.run_simulation import run_baseline_simulation

runner = run_baseline_simulation(
    network,
    duration=1000,  # 1 second
    background_rate=5.0,
    output_file="/app/results/notebook_example.h5"
)

## 3. Analyze Results

In [ ]:
from analysis.basic_analysis import (
    load_results,
    plot_raster,
    plot_firing_rates,
    plot_population_rate,
    print_summary_statistics
)

# Load results
results = load_results("/app/results/notebook_example.h5")

# Print statistics
print_summary_statistics(results)

### Spike Raster

In [ ]:
plot_raster(results['spikes'], cell_types=network.cell_types, figsize=(14, 6))

### Firing Rate Distribution

In [ ]:
plot_firing_rates(
    results['spikes'],
    results['duration'],
    cell_types=network.cell_types,
    figsize=(12, 5)
)

### Population Activity

In [ ]:
plot_population_rate(
    results['spikes'],
    results['duration'],
    bin_size=10.0,
    figsize=(14, 4)
)

## 4. Cancer Modification Example

In [ ]:
from cells import L23Pyramidal, PVBasket

# Build another network for comparison
cancer_network = build_l23_network(n_cells=200, seed=42)

# Apply cancer modifications
print("Applying cancer phenotype...")
for cell in cancer_network.cells:
    # Increase Nav channels (hyperexcitability)
    for sec in cell.all_sections:
        if hasattr(sec, 'gbar_nav16'):
            sec.gbar_nav16 *= 1.25  # 25% increase
    
    # Decrease K+ channels
    for sec in cell.all_sections:
        if hasattr(sec, 'gbar_kdr'):
            sec.gbar_kdr *= 0.80  # 20% decrease
    
    # Reduce inhibitory output
    if isinstance(cell, PVBasket):
        for syn in cell.synapses['GABAA']:
            syn.gmax *= 0.85

print("Cancer modifications applied!")

In [ ]:
# Run cancer simulation
cancer_runner = run_baseline_simulation(
    cancer_network,
    duration=1000,
    background_rate=5.0,
    output_file="/app/results/notebook_cancer.h5"
)

### Compare Control vs Cancer

In [ ]:
cancer_results = load_results("/app/results/notebook_cancer.h5")

# Calculate firing rates
control_rates = [len(times)/1.0 for times in results['spikes'].values()]
cancer_rates = [len(times)/1.0 for times in cancer_results['spikes'].values()]

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(control_rates, bins=20, alpha=0.7, label='Control', color='blue')
axes[0].hist(cancer_rates, bins=20, alpha=0.7, label='Cancer', color='red')
axes[0].set_xlabel('Firing Rate (Hz)')
axes[0].set_ylabel('Count')
axes[0].set_title('Firing Rate Distribution')
axes[0].legend()

data = [control_rates, cancer_rates]
axes[1].boxplot(data, labels=['Control', 'Cancer'])
axes[1].set_ylabel('Firing Rate (Hz)')
axes[1].set_title('Firing Rate Comparison')

plt.tight_layout()
plt.show()

print(f"\nControl mean rate: {np.mean(control_rates):.2f} Hz")
print(f"Cancer mean rate: {np.mean(cancer_rates):.2f} Hz")
print(f"Fold change: {np.mean(cancer_rates)/np.mean(control_rates):.2f}x")

## 5. Custom Analysis

Add your own analysis here!

In [ ]:
# Your analysis code here
